# Silvertip CTB — Modified vs Plan Profile Builder

JupyterLite / CSV-only workflow.

This notebook:
- matches wells **by Well Name only**
- uses the same CTB well list for both cases
- builds a **Modified Profile** from the original production + forecast CSVs
- builds a **Plan Profile** from `SilvertipPlansProduction.csv` + `SilvertipPlansForecast.csv`
- uses actual production through each well's last actual month, then forecast after that
- builds **600 months (50 years)** per well
- aggregates all wells by calendar month
- converts monthly oil/gas volumes to average daily rates
- plots Modified and Plan oil/gas profiles separately
- plots Modified vs Plan together with the **delta shaded red**, regardless of which profile is higher
- saves aggregate, well-level, QA, and comparison outputs as CSV files

Upload all required CSVs into the same JupyterLite folder before running.

## 1. Imports and file names

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

PROFILE_MONTHS = 600

# ------------------------------------------------------------
# EDIT THESE FILENAMES IF YOUR FILES HAVE DIFFERENT NAMES
# ------------------------------------------------------------

WELL_LIST_FILE = 'Silvertip.csv'                 # CTB well list
MODIFIED_PRODUCTION_FILE = 'Production.csv'      # original actual production
MODIFIED_FORECAST_FILE = 'Forecast.csv'          # original modified forecast
PLAN_PRODUCTION_FILE = 'SilvertipPlansProduction.csv'
PLAN_FORECAST_FILE = 'SilvertipPlansForecast.csv'

print('File assignments loaded.')

## 2. Helper functions

In [ ]:
def normalize_well_name(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = re.sub(r'\s+', ' ', value)
    return value.upper()


def parse_date(series):
    parsed = pd.to_datetime(series, errors='coerce')
    return parsed.dt.to_period('M').dt.to_timestamp()


def prepare_combo_csv(filename):
    """
    ComboCurve-style CSV:
    A = Well Name
    H = Date
    I = Oil BBL/month
    J = Gas MCF/month
    """
    raw = pd.read_csv(filename)

    if raw.shape[1] < 10:
        raise ValueError(f'{filename} does not contain at least columns A-J.')

    df = raw.iloc[:, [0, 7, 8, 9]].copy()
    df.columns = ['Well_Name', 'Date', 'Oil_BBL', 'Gas_MCF']

    df['Match_Name'] = df['Well_Name'].apply(normalize_well_name)
    df['Date'] = parse_date(df['Date'])
    df['Oil_BBL'] = pd.to_numeric(df['Oil_BBL'], errors='coerce').fillna(0)
    df['Gas_MCF'] = pd.to_numeric(df['Gas_MCF'], errors='coerce').fillna(0)

    df = df[df['Match_Name'].notna() & df['Date'].notna()].copy()

    df = (
        df.groupby(['Match_Name', 'Date'], as_index=False)[['Oil_BBL', 'Gas_MCF']]
        .sum()
    )

    return df


def build_case_profile(case_name, production_df, forecast_df, ctb_wells, profile_months=600):
    all_profiles = []
    qa_rows = []

    for _, well_row in ctb_wells.iterrows():
        original_name = well_row['Well_Name']
        match_name = well_row['Match_Name']

        prod_well = production_df[production_df['Match_Name'] == match_name].copy().sort_values('Date')
        fcst_well = forecast_df[forecast_df['Match_Name'] == match_name].copy().sort_values('Date')

        found_prod = len(prod_well) > 0
        found_fcst = len(fcst_well) > 0

        if found_prod:
            profile_start = prod_well['Date'].min()
        elif found_fcst:
            profile_start = fcst_well['Date'].min()
        else:
            qa_rows.append({
                'Case': case_name,
                'Well_Name': original_name,
                'Found_Production': False,
                'Found_Forecast': False,
                'Profile_Start': pd.NaT,
                'Last_Actual_Month': pd.NaT,
                'First_Forecast_Month_Used': pd.NaT,
                'Final_Profile_Months': 0,
                'Oil_BBL_50yr': 0,
                'Gas_MCF_50yr': 0,
                'QA_Status': 'MISSING FROM BOTH FILES'
            })
            continue

        monthly_dates = pd.date_range(start=profile_start, periods=profile_months, freq='MS')
        profile = pd.DataFrame({'Date': monthly_dates})
        profile['Well_Name'] = original_name
        profile['Case'] = case_name

        last_actual = prod_well['Date'].max() if found_prod else pd.NaT

        prod_temp = prod_well[['Date', 'Oil_BBL', 'Gas_MCF']].rename(columns={
            'Oil_BBL': 'Actual_Oil_BBL',
            'Gas_MCF': 'Actual_Gas_MCF'
        })
        fcst_temp = fcst_well[['Date', 'Oil_BBL', 'Gas_MCF']].rename(columns={
            'Oil_BBL': 'Forecast_Oil_BBL',
            'Gas_MCF': 'Forecast_Gas_MCF'
        })

        profile = profile.merge(prod_temp, on='Date', how='left')
        profile = profile.merge(fcst_temp, on='Date', how='left')

        if found_prod:
            actual_mask = profile['Date'] <= last_actual
            forecast_mask = profile['Date'] > last_actual
        else:
            actual_mask = pd.Series(False, index=profile.index)
            forecast_mask = pd.Series(True, index=profile.index)

        profile['Oil_BBL'] = 0.0
        profile['Gas_MCF'] = 0.0
        profile['Data_Source'] = ''

        profile.loc[actual_mask, 'Oil_BBL'] = profile.loc[actual_mask, 'Actual_Oil_BBL'].fillna(0)
        profile.loc[actual_mask, 'Gas_MCF'] = profile.loc[actual_mask, 'Actual_Gas_MCF'].fillna(0)
        profile.loc[actual_mask, 'Data_Source'] = 'Actual'

        profile.loc[forecast_mask, 'Oil_BBL'] = profile.loc[forecast_mask, 'Forecast_Oil_BBL'].fillna(0)
        profile.loc[forecast_mask, 'Gas_MCF'] = profile.loc[forecast_mask, 'Forecast_Gas_MCF'].fillna(0)
        profile.loc[forecast_mask, 'Data_Source'] = 'Forecast'

        first_forecast_used = last_actual + pd.offsets.MonthBegin(1) if found_prod else profile_start

        missing_fcst_after_actual = False
        if found_prod:
            required_dates = set(profile.loc[profile['Date'] > last_actual, 'Date'])
            fcst_dates_available = set(fcst_well['Date'])
            if not required_dates.issubset(fcst_dates_available):
                missing_fcst_after_actual = True

        if not found_fcst:
            status = 'NO FORECAST FOUND'
        elif missing_fcst_after_actual:
            status = 'FORECAST DOES NOT COVER FULL 50 YEARS'
        elif not found_prod:
            status = 'NO ACTUALS - FORECAST ONLY'
        else:
            status = 'OK'

        qa_rows.append({
            'Case': case_name,
            'Well_Name': original_name,
            'Found_Production': found_prod,
            'Found_Forecast': found_fcst,
            'Profile_Start': profile_start,
            'Last_Actual_Month': last_actual,
            'First_Forecast_Month_Used': first_forecast_used,
            'Final_Profile_Months': len(profile),
            'Oil_BBL_50yr': profile['Oil_BBL'].sum(),
            'Gas_MCF_50yr': profile['Gas_MCF'].sum(),
            'QA_Status': status
        })

        all_profiles.append(
            profile[['Case', 'Well_Name', 'Date', 'Data_Source', 'Oil_BBL', 'Gas_MCF']]
        )

    if len(all_profiles) == 0:
        raise ValueError(f'No wells matched for {case_name}.')

    well_level = pd.concat(all_profiles, ignore_index=True)
    qa = pd.DataFrame(qa_rows)

    aggregated = (
        well_level.groupby('Date', as_index=False)[['Oil_BBL', 'Gas_MCF']]
        .sum()
        .sort_values('Date')
    )

    aggregated['Days_In_Month'] = aggregated['Date'].dt.days_in_month
    aggregated['Oil_BPD'] = aggregated['Oil_BBL'] / aggregated['Days_In_Month']
    aggregated['Gas_MCFD'] = aggregated['Gas_MCF'] / aggregated['Days_In_Month']
    aggregated['Cum_Oil_BBL'] = aggregated['Oil_BBL'].cumsum()
    aggregated['Cum_Gas_MCF'] = aggregated['Gas_MCF'].cumsum()
    aggregated['Case'] = case_name

    return well_level, qa, aggregated


## 3. Load CTB well list

In [ ]:
well_list_raw = pd.read_csv(WELL_LIST_FILE)

well_name_candidates = [
    c for c in well_list_raw.columns
    if 'well' in str(c).lower() and 'name' in str(c).lower()
]

if len(well_name_candidates) > 0:
    well_list_name_col = well_name_candidates[0]
elif well_list_raw.shape[1] >= 3:
    well_list_name_col = well_list_raw.columns[2]
else:
    raise ValueError('Could not determine the Well Name column in the CTB well list.')

ctb_wells = well_list_raw[[well_list_name_col]].copy()
ctb_wells.columns = ['Well_Name']
ctb_wells['Well_Name'] = ctb_wells['Well_Name'].astype(str).str.strip()
ctb_wells = ctb_wells[
    ctb_wells['Well_Name'].notna()
    & (ctb_wells['Well_Name'] != '')
    & (ctb_wells['Well_Name'].str.lower() != 'nan')
].copy()
ctb_wells['Match_Name'] = ctb_wells['Well_Name'].apply(normalize_well_name)
ctb_wells = ctb_wells.drop_duplicates('Match_Name').reset_index(drop=True)

print('Using well name column:', well_list_name_col)
print('CTB wells found:', len(ctb_wells))
display(ctb_wells.head())

## 4. Load Modified and Plan production / forecast CSVs

In [ ]:
modified_production = prepare_combo_csv(MODIFIED_PRODUCTION_FILE)
modified_forecast = prepare_combo_csv(MODIFIED_FORECAST_FILE)
plan_production = prepare_combo_csv(PLAN_PRODUCTION_FILE)
plan_forecast = prepare_combo_csv(PLAN_FORECAST_FILE)

print('Modified production rows:', len(modified_production))
print('Modified forecast rows:', len(modified_forecast))
print('Plan production rows:', len(plan_production))
print('Plan forecast rows:', len(plan_forecast))

## 5. Build Modified Profile

In [ ]:
modified_well_level, modified_qa, modified_aggregated = build_case_profile(
    'Modified Profile',
    modified_production,
    modified_forecast,
    ctb_wells,
    PROFILE_MONTHS
)

print('Modified wells included:', modified_well_level['Well_Name'].nunique())
print(modified_qa['QA_Status'].value_counts(dropna=False))

## 6. Build Plan Profile

In [ ]:
plan_well_level, plan_qa, plan_aggregated = build_case_profile(
    'Plan Profile',
    plan_production,
    plan_forecast,
    ctb_wells,
    PROFILE_MONTHS
)

print('Plan wells included:', plan_well_level['Well_Name'].nunique())
print(plan_qa['QA_Status'].value_counts(dropna=False))

## 7. Plot Modified Profile by itself

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(modified_aggregated['Date'], modified_aggregated['Oil_BPD'], label='Modified Profile', linewidth=2)
plt.title('Silvertip CTB - Modified Oil Profile')
plt.xlabel('Date')
plt.ylabel('Oil Rate (BPD)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(modified_aggregated['Date'], modified_aggregated['Gas_MCFD'], label='Modified Profile', linewidth=2)
plt.title('Silvertip CTB - Modified Gas Profile')
plt.xlabel('Date')
plt.ylabel('Gas Rate (MCF/D)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 8. Plot Plan Profile by itself

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(plan_aggregated['Date'], plan_aggregated['Oil_BPD'], label='Plan Profile', linewidth=2)
plt.title('Silvertip CTB - Plan Oil Profile')
plt.xlabel('Date')
plt.ylabel('Oil Rate (BPD)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(plan_aggregated['Date'], plan_aggregated['Gas_MCFD'], label='Plan Profile', linewidth=2)
plt.title('Silvertip CTB - Plan Gas Profile')
plt.xlabel('Date')
plt.ylabel('Gas Rate (MCF/D)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 9. Align Modified and Plan profiles by calendar month

In [ ]:
comparison = pd.merge(
    modified_aggregated[['Date', 'Oil_BPD', 'Gas_MCFD', 'Oil_BBL', 'Gas_MCF']],
    plan_aggregated[['Date', 'Oil_BPD', 'Gas_MCFD', 'Oil_BBL', 'Gas_MCF']],
    on='Date',
    how='outer',
    suffixes=('_Modified', '_Plan')
).sort_values('Date').reset_index(drop=True)

value_columns = [
    'Oil_BPD_Modified', 'Oil_BPD_Plan',
    'Gas_MCFD_Modified', 'Gas_MCFD_Plan',
    'Oil_BBL_Modified', 'Oil_BBL_Plan',
    'Gas_MCF_Modified', 'Gas_MCF_Plan'
]

comparison[value_columns] = comparison[value_columns].fillna(0)

comparison['Oil_Delta_BPD'] = comparison['Oil_BPD_Modified'] - comparison['Oil_BPD_Plan']
comparison['Gas_Delta_MCFD'] = comparison['Gas_MCFD_Modified'] - comparison['Gas_MCFD_Plan']
comparison['Oil_Delta_BBL'] = comparison['Oil_BBL_Modified'] - comparison['Oil_BBL_Plan']
comparison['Gas_Delta_MCF'] = comparison['Gas_MCF_Modified'] - comparison['Gas_MCF_Plan']

display(comparison.head(12))

## 10. Modified vs Plan — Oil with red delta shading

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    comparison['Date'],
    comparison['Oil_BPD_Modified'],
    label='Modified Profile',
    linewidth=2
)

plt.plot(
    comparison['Date'],
    comparison['Oil_BPD_Plan'],
    label='Plan Profile',
    linewidth=2
)

plt.fill_between(
    comparison['Date'],
    comparison['Oil_BPD_Modified'],
    comparison['Oil_BPD_Plan'],
    color='red',
    alpha=0.20,
    label='Delta'
)

plt.title('Silvertip CTB - Modified vs Plan Oil Profile')
plt.xlabel('Date')
plt.ylabel('Oil Rate (BPD)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 11. Modified vs Plan — Gas with red delta shading

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    comparison['Date'],
    comparison['Gas_MCFD_Modified'],
    label='Modified Profile',
    linewidth=2
)

plt.plot(
    comparison['Date'],
    comparison['Gas_MCFD_Plan'],
    label='Plan Profile',
    linewidth=2
)

plt.fill_between(
    comparison['Date'],
    comparison['Gas_MCFD_Modified'],
    comparison['Gas_MCFD_Plan'],
    color='red',
    alpha=0.20,
    label='Delta'
)

plt.title('Silvertip CTB - Modified vs Plan Gas Profile')
plt.xlabel('Date')
plt.ylabel('Gas Rate (MCF/D)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Cumulative comparison

The final delta percentage is calculated as **(Modified - Plan) / Plan × 100**. Positive means Modified is higher; negative means Modified is lower.

In [ ]:
# Build cumulative volumes on the aligned calendar-month comparison
comparison['Cum_Oil_BBL_Modified'] = comparison['Oil_BBL_Modified'].cumsum()
comparison['Cum_Oil_BBL_Plan'] = comparison['Oil_BBL_Plan'].cumsum()
comparison['Cum_Gas_MCF_Modified'] = comparison['Gas_MCF_Modified'].cumsum()
comparison['Cum_Gas_MCF_Plan'] = comparison['Gas_MCF_Plan'].cumsum()

comparison['Cum_Oil_Delta_BBL'] = comparison['Cum_Oil_BBL_Modified'] - comparison['Cum_Oil_BBL_Plan']
comparison['Cum_Gas_Delta_MCF'] = comparison['Cum_Gas_MCF_Modified'] - comparison['Cum_Gas_MCF_Plan']

final_oil_modified = comparison['Cum_Oil_BBL_Modified'].iloc[-1]
final_oil_plan = comparison['Cum_Oil_BBL_Plan'].iloc[-1]
final_gas_modified = comparison['Cum_Gas_MCF_Modified'].iloc[-1]
final_gas_plan = comparison['Cum_Gas_MCF_Plan'].iloc[-1]

final_oil_delta = final_oil_modified - final_oil_plan
final_gas_delta = final_gas_modified - final_gas_plan

final_oil_delta_pct = (final_oil_delta / final_oil_plan * 100) if final_oil_plan != 0 else np.nan
final_gas_delta_pct = (final_gas_delta / final_gas_plan * 100) if final_gas_plan != 0 else np.nan

print(f'Oil: Modified={final_oil_modified:,.0f} BBL | Plan={final_oil_plan:,.0f} BBL | Delta={final_oil_delta:+,.0f} BBL ({final_oil_delta_pct:+.2f}%)')
print(f'Gas: Modified={final_gas_modified:,.0f} MCF | Plan={final_gas_plan:,.0f} MCF | Delta={final_gas_delta:+,.0f} MCF ({final_gas_delta_pct:+.2f}%)')

## Cumulative oil — Modified vs Plan

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(comparison['Date'], comparison['Cum_Oil_BBL_Modified'], label='Modified Profile', linewidth=2)
plt.plot(comparison['Date'], comparison['Cum_Oil_BBL_Plan'], label='Plan Profile', linewidth=2)
plt.fill_between(
    comparison['Date'],
    comparison['Cum_Oil_BBL_Modified'],
    comparison['Cum_Oil_BBL_Plan'],
    color='red', alpha=0.20, label='Cumulative Delta'
)
oil_text = f'End Delta: {final_oil_delta:+,.0f} BBL\nDelta vs Plan: {final_oil_delta_pct:+.2f}%'
plt.annotate(
    oil_text,
    xy=(comparison['Date'].iloc[-1], max(final_oil_modified, final_oil_plan)),
    xytext=(-180, -25), textcoords='offset points',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.85),
    arrowprops=dict(arrowstyle='->')
)
plt.title('Silvertip CTB - Cumulative Oil: Modified vs Plan')
plt.xlabel('Date')
plt.ylabel('Cumulative Oil (BBL)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Cumulative gas — Modified vs Plan

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(comparison['Date'], comparison['Cum_Gas_MCF_Modified'], label='Modified Profile', linewidth=2)
plt.plot(comparison['Date'], comparison['Cum_Gas_MCF_Plan'], label='Plan Profile', linewidth=2)
plt.fill_between(
    comparison['Date'],
    comparison['Cum_Gas_MCF_Modified'],
    comparison['Cum_Gas_MCF_Plan'],
    color='red', alpha=0.20, label='Cumulative Delta'
)
gas_text = f'End Delta: {final_gas_delta:+,.0f} MCF\nDelta vs Plan: {final_gas_delta_pct:+.2f}%'
plt.annotate(
    gas_text,
    xy=(comparison['Date'].iloc[-1], max(final_gas_modified, final_gas_plan)),
    xytext=(-180, -25), textcoords='offset points',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.85),
    arrowprops=dict(arrowstyle='->')
)
plt.title('Silvertip CTB - Cumulative Gas: Modified vs Plan')
plt.xlabel('Date')
plt.ylabel('Cumulative Gas (MCF)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Final cumulative delta summary

In [ ]:
cumulative_summary = pd.DataFrame({
    'Stream': ['Oil', 'Gas'],
    'Modified_Total': [final_oil_modified, final_gas_modified],
    'Plan_Total': [final_oil_plan, final_gas_plan],
    'Delta_Modified_minus_Plan': [final_oil_delta, final_gas_delta],
    'Delta_Percent_vs_Plan': [final_oil_delta_pct, final_gas_delta_pct],
    'Units': ['BBL', 'MCF']
})
display(cumulative_summary)
cumulative_summary.to_csv('Silvertip_Cumulative_Delta_Summary.csv', index=False)
comparison.to_csv('Silvertip_Modified_vs_Plan_Comparison.csv', index=False)

## 12. QA review

In [ ]:
print('Modified QA issues:')
display(modified_qa[modified_qa['QA_Status'] != 'OK'])

print('Plan QA issues:')
display(plan_qa[plan_qa['QA_Status'] != 'OK'])

## 13. Save all CSV outputs

In [ ]:
modified_aggregated.to_csv('Silvertip_Modified_Aggregated_50yr.csv', index=False)
modified_well_level.to_csv('Silvertip_Modified_Well_Level_50yr.csv', index=False)
modified_qa.to_csv('Silvertip_Modified_QA.csv', index=False)

plan_aggregated.to_csv('Silvertip_Plan_Aggregated_50yr.csv', index=False)
plan_well_level.to_csv('Silvertip_Plan_Well_Level_50yr.csv', index=False)
plan_qa.to_csv('Silvertip_Plan_QA.csv', index=False)

comparison.to_csv('Silvertip_Modified_vs_Plan_Comparison.csv', index=False)

print('Saved:')
print('  Silvertip_Modified_Aggregated_50yr.csv')
print('  Silvertip_Modified_Well_Level_50yr.csv')
print('  Silvertip_Modified_QA.csv')
print('  Silvertip_Plan_Aggregated_50yr.csv')
print('  Silvertip_Plan_Well_Level_50yr.csv')
print('  Silvertip_Plan_QA.csv')
print('  Silvertip_Modified_vs_Plan_Comparison.csv')